In [ ]:
from bs4 import BeautifulSoup

In [ ]:
# Extraction de la date


def extraire_date(fichier):

    with open(fichier, "r", encoding="utf8") as f:
        contenu = f.read()

    soup = BeautifulSoup(contenu, "html.parser")

    titre_page = soup.title.text

    parties = titre_page.split(">")

    date = parties[0].strip()

    return date




In [ ]:
# Extraction du titre


def extraire_titre(fichier):

    with open(fichier, "r", encoding="utf8") as f:
        contenu = f.read()

    soup = BeautifulSoup(contenu, "html.parser")

    titre_page = soup.title.text

    parties = titre_page.split(">")

    titre = parties[2].strip()

    return titre


In [ ]:
#Extraction du numéro du bulletin


def extraire_numéroBul(fichier):

    with open(fichier, "r", encoding="utf8") as f:
        contenu = f.read()

    soup = BeautifulSoup(contenu, "html.parser")

    titre_page = soup.title.text

    parties = titre_page.split(">")

    bulletin = parties[1].strip()

    numero_bulletin = bulletin.split()[-1]
    
    return numero_bulletin


In [ ]:
#Extraction du numéro de l'article


def extraire_numéroArticle(fichier):
    un = fichier.replace(".htm","")
    numeroArticle = un.replace("BULLETINS/", "")
    
    return numeroArticle


In [ ]:

def recuperation_rubrique(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    span = all_span[45]
    resultat = span.get_text()
    return resultat

In [ ]:
def recuperation_texte(fichier):
    try :
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # récuperer l'element 
        texte = ""
        racine = soup.body.div.table
        all_tr_niveau1 = racine.find_all("tr")
        tr_niveau1 = all_tr_niveau1[6]
        all_tr_niveau2 = tr_niveau1.td.table.find_all("tr")
        tr_niveau2 = all_tr_niveau2[2].td
        all_span = tr_niveau2.find_all("span")
        # recuperer tout les span. l'element qui caractérise le fait qu'un span soit un texte  est sa class qui est = style95
        for span in all_span:

            if span.get("class") == ["style95"]:
                texte = span.get_text() + " " + texte

        return texte
    
    except Exception as e:
        print("fichier", fichier , "erreur: ", e)
        return None

In [ ]:
def recuperation_auteur(fichier):
    # obtenir le code html de la page
    with open(fichier, "r", encoding = "UTF8") as f :
        html = f.read()

    # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
    soup = BeautifulSoup(html, 'html.parser' )
    type(soup)

    # récuperer l'element 

    racine = soup.body.div.table
    all_span = racine.find_all("span")
    ligne = 0
    # parcourir le tableau all span jusqu'a trouver index du texte rédacteur : l'auteur est situé à l'indice suivant 
    for index, span in enumerate(all_span):
        if span.get_text().strip() == "Rédacteur :" or span.get_text().strip() == "Rédacteurs :" :
            ligne = index + 1
            break
    if ligne != 0:
        span = all_span[ligne]
        texte = span.get_text()
        traitement = [x.strip() for x in texte.split("-")]
        # print ("texte : ", texte)
        # print(traitement)
        # ça ne fonctionne plus si on change de redacteur, du coup faut essayer de revoir 
        if len(traitement)>2 :
            resultat = traitement[1] + "-" + traitement[2]
            return resultat
        else :
            print ("traitement < 2")
            return fichier
    print ("pas de ligne")
    return fichier

In [ ]:
def recuperation_images(fichier):

    try :
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # récuperer l'element 
        images = {}
        racine = soup.body.div.table
        all_tr_niveau1 = racine.find_all("tr")
        tr_niveau1 = all_tr_niveau1[6]
        all_tr_niveau2 = tr_niveau1.td.table.find_all("tr")
        tr_niveau2 = all_tr_niveau2[2].td
        all_div = tr_niveau2.find_all("div")
        # recuperer les images, les mettres dans un dictionnaire. chaque dictionnaire pocede une un url et une description
        for index, div in enumerate(all_div):
            images[f"image_{index}"] = {"url" : div.img.get("src"), "description": div.span.get_text()}

        return images
    except Exception as e:
        print("fichier", fichier , "erreur: ", e)
        return None
        
fichier = "BULLETINS/67937.htm"
images = recuperation_images(fichier)
image = images["image_0"] 
print(images)
print(type(str(image["url"])))


In [ ]:


def formationxml(fichier):
    date=extraire_date(fichier)
    numéroBul=extraire_numéroBul(fichier)
    numéroArticle=extraire_numéroArticle(fichier)
    titre = extraire_titre(fichier)
    rubrique = recuperation_rubrique(fichier)
    texte = recuperation_texte(fichier)
    auteur = recuperation_auteur(fichier)
    images = recuperation_images(fichier)
    
    xml = "<document>\n"
    xml += "<date>" + date + "</date>\n"
    xml += "<bulletin>" + numéroBul + "</bulletin>\n"
    xml += "<article>" + numéroArticle + "</article>\n"
    xml += "<titre>" + titre + "</titre>\n"
    xml += "<rubrique>" + rubrique + "</rubrique>\n"
    xml += "<texte>" + texte + "</texte>\n"
    xml += "<auteur>" + auteur + "</auteur>\n"
    if len(images) != 0 :
        xml += "<images>\n"
        for image in images :
            xml += "<image>\n"
            xml += "<URL_image>" + str(image["url"]) +  "<URL_image>\n"
            xml += "<description_image>" + image["description"] + "<description_image>\n"
            xml += "<image>\n"
        xml += "<images>\n"

    xml += "</document>\n"

    return xml

In [ ]:
import os

def generer_corpus_xml(dossier_source, fichier_sortie):
    xml = "<corpus>\n"

    for fichier in os.listdir(dossier_source):
        if fichier.endswith(".htm"):
            chemin = os.path.join(dossier_source, fichier)
            doc = formationxml(chemin)
            xml += doc

    xml += "</corpus>"

    with open(fichier_sortie, "w", encoding="utf8") as f:
        f.write(xml)